In [1]:
import os
import glob
import h5py
import numpy as np
import xarray as xr
from tqdm import tqdm
import dask
from dask.diagnostics import ProgressBar

# Enable Dask
dask.config.set(scheduler='threads')  # or 'single-threaded', or 'processes'

Datasets:
- ~~G02202_V5_25km_1978_2025~~
- ~~G10016_V3_PROTO_25km_2025~~
- ~~NSIDC_0803_25km_2023_Present~~
- ~~AMSR2_25km_2012_Present~~
- ~~AMSR2_12.5km_2012_Present~~

### G02202

In [7]:
# Dataset and base path
dataset = 'G02202_V5_25km_1978_2025'
base_path = f'/glade/work/skygale/data/{dataset}/'
all_files = sorted(glob.glob(os.path.join(base_path, '*', '*.nc')))
print(f"🔍 Found {len(all_files)} NetCDF files.")

datasets = []

for f in tqdm(all_files, desc="Loading NetCDF files with Dask (lazy)"):
    try:
        ds = xr.open_dataset(f, chunks={'time': 1})
        datasets.append(ds)
    except Exception as e:
        print(f"❌ Skipping unreadable file: {f}\n   Error: {e}")

# Manually concatenate
combined_ds = xr.concat(datasets, dim='time', combine_attrs='override')

# Sort by time (check)
if 'time' in combined_ds.coords:
    combined_ds = combined_ds.sortby('time')

# Compression encoding
encoding = {
    var: {"zlib": True, "complevel": 4}
    for var in combined_ds.data_vars
}

# Output path
output_path = '/glade/work/skygale/data/combined/G02202_25km_1978_2025.nc'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Use Dask to write lazily, with progress bar
with ProgressBar():
    combined_ds.to_netcdf(
        path=output_path,
        mode='w',
        format='NETCDF4',
        engine='netcdf4',
        encoding=encoding,
        compute=True
    )

print(f"\n✅ Successfully saved combined dataset to:\n{output_path}")

Loading NetCDF files with Dask (lazy): 0it [00:00, ?it/s]


ValueError: must supply at least one object to concatenate

### G10016 Prototype

In [4]:
# Dataset and base path
dataset = 'G10016_V3_PROTO_25km_2025'
base_path = f'/glade/work/skygale/pbi-data/{dataset}/'
all_files = sorted(glob.glob(os.path.join(base_path, '*.nc')))
print(f"🔍 Found {len(all_files)} NetCDF files.")

datasets = []

for f in tqdm(all_files, desc="Loading NetCDF files with Dask (lazy)"):
    try:
        ds = xr.open_dataset(f, chunks={'time': 1})
        datasets.append(ds)
    except Exception as e:
        print(f"❌ Skipping unreadable file: {f}\n   Error: {e}")

# Manually concatenate
combined_ds = xr.concat(datasets, dim='time', combine_attrs='override')

# Sort by time (check)
if 'time' in combined_ds.coords:
    combined_ds = combined_ds.sortby('time')

# Compression encoding
encoding = {
    var: {"zlib": True, "complevel": 4}
    for var in combined_ds.data_vars
}

# Output path
output_path = '/glade/work/skygale/pbi-data/combined/G10016_PROTO_25km_2025.nc'
combined_ds = combined_ds.chunk({'time': 30})

# Save as combined file
os.makedirs(os.path.dirname(output_path), exist_ok=True)

print("💾 Saving combined dataset to NetCDF...")
with ProgressBar():
    combined_ds.to_netcdf(
        output_path,
        mode='w',
        format='NETCDF4',
        engine='h5netcdf',
        encoding=encoding,
        compute=True
    )

print(f"✅ Saved combined dataset to:\n{output_path}")

🔍 Found 242 NetCDF files.


Loading NetCDF files with Dask (lazy): 100%|██████████| 242/242 [00:08<00:00, 27.75it/s]


💾 Saving combined dataset to NetCDF...
[########################################] | 100% Completed | 12.99 ss
✅ Saved combined dataset to:
/glade/work/skygale/data/combined/G10016_PROTO_25km_2025.nc


### NSIDC 0803

In [2]:
# Dataset and base path
dataset = 'NSIDC_0803_25km_2023_Present'
base_path = f'/glade/work/skygale/data/{dataset}/'
all_files = sorted(glob.glob(os.path.join(base_path, '*', '*', '*.nc')))
print(f"🔍 Found {len(all_files)} NetCDF files.")

datasets = []

for f in tqdm(all_files, desc="Loading NetCDF files with Dask (lazy)"):
    try:
        ds = xr.open_dataset(f, chunks={'time': 1})
        datasets.append(ds)
    except Exception as e:
        print(f"❌ Skipping unreadable file: {f}\n   Error: {e}")

# Manually concatenate
combined_ds = xr.concat(datasets, dim='time', combine_attrs='override')

# Sort by time
if 'time' in combined_ds.coords:
    combined_ds = combined_ds.sortby('time')

# Compression encoding
encoding = {
    var: {"zlib": True, "complevel": 4}
    for var in combined_ds.data_vars
}

# Output path
output_path = '/glade/work/skygale/data/combined/NSIDC_0803_25km_2023_Present.nc'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Use Dask to write lazily, with progress bar
with ProgressBar():
    combined_ds.to_netcdf(
        path=output_path,
        mode='w',
        format='NETCDF4',
        engine='netcdf4',
        encoding=encoding,
        compute=True
    )

print(f"\n✅ Successfully saved combined dataset to:\n{output_path}")

🔍 Found 981 NetCDF files.


Loading NetCDF files with Dask (lazy): 100%|██████████| 981/981 [00:29<00:00, 33.53it/s]


[########################################] | 100% Completed | 20.55 s

✅ Successfully saved combined dataset to:
/glade/work/skygale/data/combined/NSIDC_0803_25km_2023_Present.nc


### AMSR2 25km

In [ ]:
# Input folder with .he5 files
input_dir = '/glade/work/skygale/data/AMSR2_25km_2012_Present/'
output_path = '/glade/work/skygale/data/combined/AMSR2_25km_2012_2024.nc'

# Temp folder for per-day netcdf files
temp_dir = '/glade/work/skygale/data/temp_nc'
os.makedirs(temp_dir, exist_ok=True)

# Get all .he5 files
all_files = sorted(glob.glob(os.path.join(input_dir, '*.he5')))
print(f"Found {len(all_files)} .he5 files")

# Container for each day's data
all_data = []


def read_he5_to_xarray(file_path):
    """
    Custom function to read AMSR2 25km sea ice concentration .he5 file into xarray.DataArray.
    """
    try:
        # File structure may differ slightly — confirm these paths!
        SIC_PATH = '/HDFEOS/GRIDS/NpPolarGrid25km/Data Fields/SI_25km_NH_ICECON_DAY'
        LAT_PATH = '/HDFEOS/GRIDS/NpPolarGrid25km/lat'
        LON_PATH = '/HDFEOS/GRIDS/NpPolarGrid25km/lon'

        with h5py.File(file_path, 'r') as f:
            sic = f[SIC_PATH][:]
            lat = f[LAT_PATH][:]
            lon = f[LON_PATH][:]

        # Transpose if necessary — usually (y, x) → (ni, nj)
        sic = np.transpose(sic)
        lat = np.transpose(lat)
        lon = np.transpose(lon)

        # Extract date from filename
        fname = os.path.basename(file_path)
        date_str = fname.split('_')[-1].replace('.he5', '')  # e.g., '20230715'
        timestamp = np.datetime64(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}", 'ns')

        # Create DataArray
        ds = xr.DataArray(
            data=sic,
            dims=["ni", "nj"],
            coords={
                "lat": (["ni", "nj"], lat),
                "lon": (["ni", "nj"], lon),
                "time": timestamp
            },
            name="sic"
        )

        # Mask invalid SIC values (e.g., >100)
        ds = ds.where(ds <= 100, other=np.nan)

        # Expand time dimension
        ds = ds.expand_dims("time")

        return ds

    except Exception as e:
        print(f"⚠️ Failed to read {file_path}:\n   {e}")
        return None


# Process all files
temp_files = []
for idx, f in enumerate(tqdm(all_files, desc="Reading .he5 files")):
    ds = read_he5_to_xarray(f)
    if ds is not None:
        temp_file = os.path.join(temp_dir, f'temp_{idx:05d}.nc')
        ds.to_netcdf(temp_file)
        temp_files.append(temp_file)

if not temp_files:
    raise RuntimeError("❌ No valid data arrays were loaded.")


def open_in_batches(file_list, batch_size=500, sort_time=True, engine='netcdf4'):
    """
    Open large number of NetCDFs in safe batches and concatenate.

    Parameters:
        file_list (list): List of file paths
        batch_size (int): Max number of files per open_mfdataset
        sort_time (bool): Sort final dataset by time
        engine (str): NetCDF engine ('netcdf4', 'h5netcdf', etc.)

    Returns:
        xarray.Dataset
    """
    batches = []
    total_files = len(file_list)

    print("📚 Opening all temp NetCDFs lazily...")
    for i in range(0, total_files, batch_size):
        batch = file_list[i:i+batch_size]
        print(f"📦 Opening batch {i//batch_size + 1} with {len(batch)} files...")

        ds = xr.open_mfdataset(
            batch,
            combine='by_coords',
            parallel=False,        # Parallel doesn't help much here
            chunks={},             # Enable Dask
            engine=engine
        )
        batches.append(ds)

    print("🔗 Concatenating all batches...")
    combined = xr.concat(batches, dim='time')

    if sort_time and 'time' in combined:
        combined = combined.sortby('time')

    return combined


# Concatenate
temp_files_ = glob.glob('/glade/work/skygale/data/temp_nc/*.nc')
ds_all = open_in_batches(temp_files_, batch_size=500)

# Chunk
output_path = '/glade/work/skygale/data/combined/AMSR2_25km_2012_2024.nc'
ds_all = ds_all.chunk({'time': 30})

# Compression encoding
encoding = {
    'sic': {"zlib": True, "complevel": 1}
}

# Save as combined file
os.makedirs(os.path.dirname(output_path), exist_ok=True)

print("💾 Saving combined dataset to NetCDF...")
with ProgressBar():
    ds_all.to_netcdf(
        output_path,
        mode='w',
        format='NETCDF4',
        engine='h5netcdf',
        encoding=encoding,
        compute=True
    )

print(f"✅ Saved combined dataset to:\n{output_path}")

Found 4806 .he5 files


Reading .he5 files: 100%|██████████| 4806/4806 [15:31<00:00,  5.16it/s]


📚 Opening all temp NetCDFs lazily...


### AMSR2 12.5km

In [ ]:
# Input folder with .he5 files
input_dir = '/glade/work/skygale/data/AMSR2_12.5km_2012_Present/'
output_path = '/glade/work/skygale/data/combined/AMSR2_12.5km_2012_Present.nc'

# Temp folder for per-day netcdf files
temp_dir = '/glade/work/skygale/data/temp_nc_'
os.makedirs(temp_dir, exist_ok=True)

# Get all .he5 files
all_files = sorted(glob.glob(os.path.join(input_dir, '*.he5')))
print(f"Found {len(all_files)} .he5 files")

# Container for each day's data
all_data = []


def read_he5_to_xarray(file_path):
    """
    Custom function to read AMSR2 12.5km sea ice concentration .he5 file into xarray.DataArray.
    """
    try:
        # File structure may differ slightly — confirm these paths!
        SIC_PATH = '/HDFEOS/GRIDS/NpPolarGrid25km/Data Fields/SI_25km_NH_ICECON_DAY'
        LAT_PATH = '/HDFEOS/GRIDS/NpPolarGrid25km/lat'
        LON_PATH = '/HDFEOS/GRIDS/NpPolarGrid25km/lon'

        with h5py.File(file_path, 'r') as f:
            sic = f[SIC_PATH][:]
            lat = f[LAT_PATH][:]
            lon = f[LON_PATH][:]

        # Transpose if necessary — usually (y, x) → (ni, nj)
        sic = np.transpose(sic)
        lat = np.transpose(lat)
        lon = np.transpose(lon)

        # Extract date from filename
        fname = os.path.basename(file_path)
        date_str = fname.split('_')[-1].replace('.he5', '')  # e.g., '20230715'
        timestamp = np.datetime64(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}", 'ns')

        # Create DataArray
        ds = xr.DataArray(
            data=sic,
            dims=["ni", "nj"],
            coords={
                "lat": (["ni", "nj"], lat),
                "lon": (["ni", "nj"], lon),
                "time": timestamp
            },
            name="sic"
        )

        # Mask invalid SIC values (e.g., >100)
        ds = ds.where(ds <= 100, other=np.nan)

        # Expand time dimension
        ds = ds.expand_dims("time")

        return ds

    except Exception as e:
        print(f"⚠️ Failed to read {file_path}:\n   {e}")
        return None


# Process all files
temp_files = []
for idx, f in enumerate(tqdm(all_files, desc="Reading .he5 files")):
    ds = read_he5_to_xarray(f)
    if ds is not None:
        temp_file = os.path.join(temp_dir, f'temp_{idx:05d}.nc')
        ds.to_netcdf(temp_file)
        temp_files.append(temp_file)

if not temp_files:
    raise RuntimeError("❌ No valid data arrays were loaded.")


# Concatenate
open_in_batches(file_list, batch_size=500, sort_time=True, engine='netcdf4'):
    """
    Open large number of NetCDFs in safe batches and concatenate.

    Parameters:
        file_list (list): List of file paths
        batch_size (int): Max number of files per open_mfdataset
        sort_time (bool): Sort final dataset by time
        engine (str): NetCDF engine ('netcdf4', 'h5netcdf', etc.)

    Returns:
        xarray.Dataset
    """
    batches = []
    total_files = len(file_list)

    print("📚 Opening all temp NetCDFs lazily...")
    for i in range(0, total_files, batch_size):
        batch = file_list[i:i+batch_size]
        print(f"📦 Opening batch {i//batch_size + 1} with {len(batch)} files...")

        ds = xr.open_mfdataset(
            batch,
            combine='by_coords',
            parallel=False,        # Parallel doesn't help much here
            chunks={},             # Enable Dask
            engine=engine
        )
        batches.append(ds)

    print("🔗 Concatenating all batches...")
    combined = xr.concat(batches, dim='time')

    if sort_time and 'time' in combined:
        combined = combined.sortby('time')

    return combined


temp_files_ = glob.glob('/glade/work/skygale/data/temp_nc_/*.nc')
ds_all = open_in_batches(temp_files_, batch_size=500)

# Compression encoding
encoding = {
    'sic': {"zlib": True, "complevel": 4}
}

# Save as combined file
os.makedirs(os.path.dirname(output_path), exist_ok=True)

print("💾 Saving combined dataset to NetCDF...")
with ProgressBar():
    ds_all.to_netcdf(
        output_path,
        mode='w',
        format='NETCDF4',
        engine='netcdf4',
        encoding=encoding,
        compute=True
    )

print(f"\n✅ Saved combined dataset to:\n{output_path}")